# Latency probe: cross-hardware run

Runs this project's own `benchmarks/latency_probe.py` on a Colab GPU, so the
numbers are directly comparable to a local run. The methodology is not
reimplemented here - this notebook only clones, installs, runs, and exports.

**Before running:** Runtime > Change runtime type > T4 GPU.

Why a T4 specifically: it has tensor cores, and the development machine
(GTX 1650 Ti) does not, even though both report CUDA compute capability 7.5 -
Nvidia removed the tensor cores from the GTX 16-series. Locally, fp16 gave no
speedup and was sometimes slower. If fp16 helps on a T4, that isolates the cause
instead of leaving it an unexplained curiosity.

Runtime: roughly 10 minutes, most of it installing audiocraft.

## 1. Confirm the GPU

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU attached. Runtime > Change runtime type > T4 GPU, then rerun.")

props = torch.cuda.get_device_properties(0)
capability = torch.cuda.get_device_capability(0)

print(f"GPU        : {torch.cuda.get_device_name(0)}")
print(f"VRAM       : {props.total_memory / 1e9:.1f} GB")
print(f"Capability : {capability[0]}.{capability[1]}")
print(f"Torch      : {torch.__version__}")

## 2. Clone and install

Installing `audiocraft` may prompt a runtime restart. If it does, restart and
rerun from this cell - the clone is idempotent.

In [ ]:
import os

if not os.path.isdir("Brain-Music-Therapy"):
    !git clone -q https://github.com/kirthankulkarni-bit/Brain-Music-Therapy.git

%pip install -q audiocraft

## 3. Run the probe

Sections A and B measure the analysis path and DSP compute. No GPU is involved in
those, so they should roughly match the local run - they are included as a control.
Section C is the GPU-dependent part and is the reason for this notebook.

In [ ]:
import re

import torch

label = "colab-" + re.sub(r"[^a-z0-9]+", "-", torch.cuda.get_device_name(0).lower()).strip("-")
out = f"benchmarks/latency_{label}.json"
print(f"label: {label}")

!cd Brain-Music-Therapy && python benchmarks/latency_probe.py --label "$label" --durations 4 8 --trials 3 --out "$out"

## 4. Results

In [ ]:
import json

import pandas as pd

results = json.load(open(f"Brain-Music-Therapy/{out}"))
hw = results["hardware"]

print(f"{hw['gpu_name']}  |  capability {hw['compute_capability']}"
      f"  |  tensor cores: {hw['has_tensor_cores']}")
print(f"end-to-end worst case: {results['end_to_end_worst_case_s']:.1f} s")
print()

pd.DataFrame(results["musicgen"])[
    ["precision", "duration_s", "median_generation_s", "realtime_factor", "faster_than_realtime"]
]

## 5. Compare fp16 against fp32

The single number this notebook exists to produce. On the GTX 1650 Ti the speedup
was about 1.0x or worse; a tensor-core GPU should be clearly above 1.0x.

In [ ]:
rows = {(r["precision"], r["duration_s"]): r["median_generation_s"] for r in results["musicgen"]}
durations = sorted({d for _, d in rows})

print(f"{'duration':>9} {'fp32':>9} {'fp16':>9} {'speedup':>9}")
print("-" * 39)
for d in durations:
    fp32, fp16 = rows.get(("fp32", d)), rows.get(("fp16", d))
    if fp32 and fp16:
        print(f"{d:>8.0f}s {fp32:>8.2f}s {fp16:>8.2f}s {fp32 / fp16:>8.2f}x")

## 6. Export

Download the JSON and commit it to `benchmarks/` next to the local result. Each
machine writes its own file, so the comparison table in the paper is built from
whichever files are present.

In [ ]:
from google.colab import files

files.download(f"Brain-Music-Therapy/{out}")